# KagSSH Go — Kaggle + ChatGPT u otros agentes

**Propósito:** usar el entorno de ejecución de un cuaderno de Kaggle desde **ChatGPT u otro agente que disponga de una herramienta SSH/terminal autorizada**, para **diagnosticar y corregir cuadernos rotos**, depurar dependencias/código y **crear proyectos de machine learning y aprendizaje**.

**Aclaración importante:** este proyecto **no conecta automáticamente el chat de ChatGPT con Kaggle**, no es una integración oficial, no ofrece una API de ChatGPT ni implementa un servidor MCP. KagSSH Go crea un servidor SSH dentro de Kaggle y un **túnel SSH inverso** hacia un VPS propio. Un agente con acceso SSH/SFTP debe conectarse explícitamente por esa vía.

Este cuaderno configura el acceso remoto; **no entrena modelos ni modifica notebooks automáticamente**. El agente/usuario decide y ejecuta esas tareas posteriormente.

## 1. Arquitectura y requisitos

```text
ChatGPT/otro agente con herramienta SSH (o un desarrollador por terminal)
              |
              v
      PC/host del agente --SSH TCP 2223--> IP pública del VPS:2223
                                      ^
                                      | túnel SSH inverso desde Kaggle al SSH del VPS:22
                                      |
                       Kaggle:127.0.0.1:2224 (KagSSH Go)
                                      |
                       /kaggle/working (proyectos y .ipynb)
```

Antes de ejecutar:

1. Crea un notebook de Kaggle con **Internet habilitado**. Las reglas de cada competición pueden restringirlo.
2. No necesitas un binario adjunto. Las primeras celdas clonan el repositorio público, instalan Go si hace falta y compilan KagSSH en Kaggle.
3. Usa un **VPS propio o autorizado** con OpenSSH activo en el puerto `22`, que permita `AllowTcpForwarding yes`, `GatewayPorts clientspecified` o `yes`, y el túnel inverso al puerto público `2223`; abre ese puerto en el firewall.
4. Configura las opciones `SSH_*` mediante **Add-ons → Secrets** (recomendado para contraseñas) o variables de entorno (ver sección 3).
5. Prepara un ordenador/agente con cliente SSH/SFTP, acceso legítimo al VPS y autorización para operar el notebook.

El túnel pide **siempre** `0.0.0.0:2223` en el VPS para conectar desde la IP pública. No hay variable para cambiar ese destino a localhost. Si el VPS restringe GatewayPorts o el firewall, la IP pública no será accesible aunque el proceso arranque. Protege el acceso SSH con credenciales sólidas y límites de firewall.

## 2. Clonar el repositorio remoto y compilar en Kaggle

No necesitas subir un ejecutable. **Ejecuta las siguientes tres celdas en orden** con Internet habilitado: clonar/actualizar `main` desde GitHub (sin sobrescribir cambios locales), comprobar/descargar Go y compilar el binario directamente en `/kaggle/working`. Si Go no está instalado o es anterior al mínimo de `go.mod`, se descarga de **go.dev** con comprobación del SHA-256 publicado por el equipo de Go. El binario se reconstruye cada vez que ejecutas la celda de compilación.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import platform
import re
import shutil
import subprocess
import tempfile
import urllib.request

REPO_URL = "https://github.com/ThowiLabs/kagssh-go.git"
WORKING = Path("/kaggle/working")
REPO_DIR = WORKING / "kagssh-go"
BINARY = WORKING / "kagssh-linux-amd64"

if platform.system() != "Linux" or platform.machine().lower() not in ("x86_64", "amd64"):
    raise RuntimeError("Este ejemplo requiere Kaggle Linux x86_64 (amd64).")
if not WORKING.is_dir():
    raise RuntimeError("Ejecuta este notebook en Kaggle (/kaggle/working).")
if not shutil.which("git"):
    raise RuntimeError("No se encontró git en el entorno de Kaggle.")

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(REPO_DIR)],
        check=True, timeout=180,
    )
else:
    if not (REPO_DIR / ".git").is_dir():
        raise RuntimeError(f"{REPO_DIR} existe pero no es un clon Git. Muévelo antes de continuar.")
    current_origin = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"], text=True
    ).strip().rstrip("/")
    if current_origin.removesuffix(".git").lower() != REPO_URL.removesuffix(".git").lower():
        raise RuntimeError(f"El directorio ya existe y su remoto es distinto: {current_origin}")
    dirty = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "status", "--porcelain"], text=True
    ).strip()
    if dirty:
        raise RuntimeError("El clon tiene cambios sin guardar; no se actualizará ni se borrará nada.")
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "fetch", "origin", "main"],
        check=True, timeout=180,
    )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "merge", "--ff-only", "origin/main"],
        check=True, timeout=60,
    )

REVISION = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], text=True
).strip()
print("Repositorio listo:", REPO_DIR)
print("Revisión remota clonada:", REVISION)

In [ ]:
# Usa el Go del sistema si cumple el mínimo de go.mod; si falta o es antiguo,
# instala la versión necesaria desde la distribución oficial con SHA-256 verificado.
mod_text = (REPO_DIR / "go.mod").read_text(encoding="utf-8")
match = re.search(r"(?m)^go\s+(\d+)\.(\d+)(?:\.(\d+))?\s*$", mod_text)
if not match:
    raise RuntimeError("No se encontró la versión mínima de Go en go.mod.")
MIN_GO = tuple(int(v or 0) for v in match.groups())
required = ".".join(str(v) for v in MIN_GO)
go_executable = shutil.which("go")

def installed_version(executable):
    if not executable:
        return None
    result = subprocess.run([executable, "version"], capture_output=True, text=True)
    found = re.search(r"go(\d+)\.(\d+)\.(\d+)", result.stdout)
    return tuple(map(int, found.groups())) if result.returncode == 0 and found else None

existing = installed_version(go_executable)
# Go 1.27.0 y 1.27.1 preceden a los parches del 8/oct/2026.
def secure_go(version):
    return (version is not None and version >= MIN_GO
            and (version[:2] != (1, 27) or version[2] >= 2))

if secure_go(existing):
    GO_BIN = Path(go_executable)
    print("Go instalado suficiente:", subprocess.check_output([str(GO_BIN), "version"], text=True).strip())
else:
    if not os.environ.get("KAGGLE_USER_SECRETS_TOKEN") and not os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        print("Nota: esta instalación necesita acceso a Internet desde Kaggle.")
    release_version = "go" + required
    catalog_url = "https://go.dev/dl/?mode=json&include=all"
    with urllib.request.urlopen(catalog_url, timeout=40) as response:
        releases = json.load(response)
    release = next((item for item in releases if item.get("version") == release_version), None)
    if release is None:
        raise RuntimeError(f"La versión {release_version} no figura en las descargas oficiales.")
    archive_meta = next(
        (f for f in release["files"] if f.get("os") == "linux"
         and f.get("arch") == "amd64" and f.get("kind") == "archive"), None
    )
    if archive_meta is None or not re.fullmatch(r"[0-9a-f]{64}", archive_meta.get("sha256", "")):
        raise RuntimeError("No se pudo obtener el SHA-256 oficial de Go linux-amd64.")
    tempdir = Path(tempfile.mkdtemp(prefix="kagssh-toolchain-", dir="/tmp"))
    archive = tempdir / archive_meta["filename"]
    address = "https://go.dev/dl/" + archive_meta["filename"]
    print("Descargando Go verificado:", archive_meta["filename"])
    digest = hashlib.sha256()
    with urllib.request.urlopen(address, timeout=120) as source, archive.open("wb") as target:
        while chunk := source.read(1024 * 1024):
            target.write(chunk)
            digest.update(chunk)
    if digest.hexdigest() != archive_meta["sha256"]:
        archive.unlink(missing_ok=True)
        raise RuntimeError("SHA-256 inválido: se rechaza la instalación.")
    subprocess.run(["tar", "-xzf", str(archive), "-C", str(tempdir)], check=True, timeout=120)
    archive.unlink(missing_ok=True)
    GO_BIN = tempdir / "go" / "bin" / "go"
    if not GO_BIN.is_file() or installed_version(str(GO_BIN)) != MIN_GO:
        raise RuntimeError("No se pudo validar la versión instalada de Go.")

print("Go seleccionado:", subprocess.check_output([str(GO_BIN), "version"], text=True).strip())
print("Versión mínima del repositorio:", required)

In [ ]:
# Compila el código clonado para el runtime Linux de Kaggle (sin CGO).
build_env = os.environ.copy()
build_env.update({"CGO_ENABLED": "0", "GOOS": "linux", "GOARCH": "amd64", "GOTOOLCHAIN": "local"})
subprocess.run(
    [str(GO_BIN), "build", "-buildvcs=false", "-trimpath", "-ldflags=-s -w",
     "-o", str(BINARY), "./cmd/kagssh"],
    cwd=str(REPO_DIR), env=build_env, check=True, timeout=600,
)
version = subprocess.run([str(BINARY), "-version"], capture_output=True, text=True, check=True)
print("Compilación terminada:", BINARY)
print("Versión binaria:", version.stdout.strip())
print("Go usado:", subprocess.check_output([str(GO_BIN), "version"], text=True).strip())

## 3. Configuración: variables de entorno o Kaggle Secrets

**Hay dos formas equivalentes de configurar KagSSH:** crea en Kaggle **Add-ons → Secrets** una entrada por etiqueta exacta `SSH_*`, y autorízala para este notebook; o define las mismas variables como variables de entorno del proceso. **Prioridad:** variable de entorno explícita → Secret Kaggle del mismo nombre → valor por defecto. El notebook no imprime ni necesita conocer el contenido de los Secrets. Para credenciales, usa Secrets, no contraseñas embebidas en celdas públicas.

| Variable o nombre del Secret | Valor por defecto | Uso |
|---|---|---|
| `SSH_HOST` | **obligatorio** | IP pública o dominio de tu VPS |
| `SSH_USER` | `root` | Usuario SSH del VPS |
| `SSH_PASSWORD` | vacío | Contraseña del VPS; alternativa: `SSH_KEY` |
| `SSH_KEY` | vacío | **Ruta** a la clave privada para acceder al VPS, no contenido PEM |
| `SSH_FINGERPRINT` | vacío | Opcional, preferido si conoces la huella verificada del VPS |
| `SSH_KNOWN_HOSTS` | `~/.ssh/known_hosts` | Se crea automáticamente si falta, guardando la primera clave del VPS |
| `SSH_PORT_REMOTE` | `22` | Puerto SSH **del VPS** |
| `SSH_PORT_KAGGLE` | `2223` | Puerto **publicado en el VPS** por el túnel inverso |
| `SSH_PORT_LOCAL` | `2224` | Puerto de SSH dentro de Kaggle |
| `SSH_LOGIN_USER` | usuario Linux del proceso | Usuario de login del SSH incorporado en Kaggle |
| `SSH_LOGIN_PASSWORD` | vacío | Contraseña **distinta** para entrar a Kaggle |
| `SSH_AUTHORIZED_KEYS` | vacío | **Ruta** a archivo con claves públicas autorizadas, alternativa al login por contraseña |
| `SSH_HOST_KEY` | `~/.config/kagssh/host_ed25519` | Ruta a la clave de host persistente en Kaggle |

**Mínimos prácticos**: define `SSH_HOST`, `SSH_USER`, `SSH_PASSWORD` (o `SSH_KEY`) y `SSH_LOGIN_PASSWORD` (o `SSH_AUTHORIZED_KEYS`). `SSH_LOGIN_USER` es opcional pero conviene definirlo para saber con qué nombre iniciar sesión. No necesitas aportar `SSH_FINGERPRINT` ni crear `known_hosts` para arrancar. En ausencia de huella verificada, KagSSH registra automáticamente la primera clave del VPS; **ese primer contacto no está autenticado independientemente**, por lo que debes comprobar después la huella para descartar suplantación.

**Alternativa con variables:** antes de ejecutar `-check` y el binario, define variables `SSH_*` **fuera de un notebook público**, en el entorno del proceso. Al usar Kaggle Secrets, **no necesitas escribir `export`**. `SSH_KEY` y `SSH_AUTHORIZED_KEYS` admiten rutas de archivos existentes, no materializan automáticamente claves guardadas como texto Secret.

Comprueba la huella del VPS mediante un canal seguro, por ejemplo con `ssh-keygen -lf /etc/ssh/ssh_host_ed25519_key.pub -E sha256` desde el propio VPS. No desactives la comprobación de host.

**Acceso externo:** el puerto remoto se solicita **siempre en `0.0.0.0`**, no en localhost. Configura el SSH del VPS para permitir `GatewayPorts clientspecified` (o `yes`) y abre el puerto TCP `2223` en su firewall.

**Limitación actual:** la lectura de Secrets y el túnel completo necesitan aún una prueba real con Kaggle y un VPS autorizado. `-check` solo comprueba la configuración.

## 4. Validar Secrets y configuración

La celda siguiente prueba que el binario compilado acepta los parámetros, sin establecer todavía una conexión SSH. Configura antes tus Secrets o variables `SSH_*`.

In [ ]:
# Comprueba que el binario puede leer la configuración, sin abrir el túnel.
check = subprocess.run(
    [str(BINARY), "-check"],
    text=True,
    capture_output=True,
    timeout=180,
)
if check.returncode != 0:
    # El programa no debería incluir los valores de Secrets en sus errores.
    # Revisa la configuración y evita compartir capturas/logs privados.
    raise RuntimeError(
        "Configuración inválida (o error al consultar Kaggle Secrets). "
        "Revisa etiquetas, autorización, Internet y huella del VPS. "
        f"Detalle: {check.stderr.strip()[:600]}"
    )
print(check.stdout.strip())
print("La validación NO garantiza que el VPS acepte el túnel.")

## 5. Ejecuta KagSSH y deténlo desde Kaggle

**Ejecuta la siguiente celda normalmente.** Permanecerá activa mientras KagSSH esté funcionando: Kaggle mostrará el botón de **Detener/Interruptar** y los logs aparecerán directamente en la salida de esta misma celda, en tiempo real.

Para cerrar la sesión SSH y el túnel inverso, pulsa **Detener/Interruptar** en esa celda. No se inicia ningún proceso en segundo plano ni se usan visores de logs, hilos, archivos intermedios o una celda STOP adicional.

Si Kaggle termina el runtime, la conexión SSH también se pierde.

In [ ]:
# Mantiene la celda de Kaggle en ejecución y muestra los logs de Go directamente.
# Usa Detener / Interruptar celda en Kaggle para cerrar KagSSH.
import subprocess

print("Iniciando KagSSH. Pulsa Detener/Interruptar en Kaggle para desconectar.", flush=True)
with subprocess.Popen(
    [str(BINARY)],
    cwd=str(WORKING),
    stdin=subprocess.DEVNULL,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    errors="replace",
    # Se reenvía cada línea a print() de Jupyter, sin hilos ni segundo plano.
) as kagssh:
    try:
        for line in kagssh.stdout:  # Bucle bloqueante: mantiene activa la celda.
            print(line, end="", flush=True)  # Logs visibles en la misma celda.
        status = kagssh.wait()
    except KeyboardInterrupt:
        print("\nInterrupción de Kaggle: cerrando KagSSH...", flush=True)
        if kagssh.poll() is None:
            kagssh.terminate()
        try:
            kagssh.wait(timeout=10)
        except subprocess.TimeoutExpired:
            kagssh.kill()
            kagssh.wait()
        print("KagSSH detenido; túnel cerrado.", flush=True)
    else:
        if status != 0:
            raise RuntimeError(f"KagSSH terminó con error (código {status}). Revisa los logs anteriores.")
        print("KagSSH terminó normalmente.", flush=True)

## 6. Conecta directamente desde tu PC o desde el host del agente

**Estos comandos se ejecutan FUERA de Kaggle**, desde tu ordenador o desde el entorno autorizado del agente. No tienes que iniciar sesión primero en el VPS.

El puerto público predeterminado es `2223` y el usuario es el valor de `SSH_LOGIN_USER`:

```bash
ssh -p 2223 usuario_kaggle@IP_PUBLICA_DEL_VPS
# Para transferir ficheros:
sftp -P 2223 usuario_kaggle@IP_PUBLICA_DEL_VPS
```

Sustituye `IP_PUBLICA_DEL_VPS` por tu IP real y usa el nombre de usuario configurado en KagSSH. Cambia `2223` solo si configuraste otro `SSH_PORT_KAGGLE`. Verifica la huella SSH que muestre el cliente antes de confiar en el acceso.

**Configuración indispensable del VPS:** su servidor OpenSSH debe permitir `AllowTcpForwarding yes` y `GatewayPorts clientspecified` (o `yes`), y el firewall debe permitir **TCP 2223**. Si `GatewayPorts` está desactivado, el VPS puede forzar localhost aunque KagSSH pida `0.0.0.0`. No puede cambiarse ese ajuste del VPS desde Kaggle.

**Seguridad:** el puerto SSH de Kaggle quedará expuesto a Internet; usa credenciales robustas, preferentemente claves, y restringe los orígenes permitidos por el firewall.

**Uso con ChatGPT/agentes:** el agente necesita una **herramienta de terminal/SSH** que pueda alcanzar tu VPS. El chat no se conecta por sí solo a Kaggle y KagSSH no incorpora un servidor MCP.

## 7. Qué puede hacer el agente después de conectar

- **Reparar un notebook roto:** revisar ficheros `.ipynb`, dependencias, errores de ejecución, rutas y resultados, editar una **copia** en `/kaggle/working` y probarla.
- **Crear proyectos de machine learning:** generar scripts, preparar datos, ejecutar entrenamientos y evaluar modelos con los recursos disponibles en la sesión de Kaggle.
- **Aprender y experimentar:** pedir explicaciones, refactorizaciones y pruebas guiadas por un agente; SSH ofrece el acceso al entorno, no sustituye el razonamiento del agente.

Ejemplo de comandos desde la sesión SSH **ya conectada**:

```bash
cd /kaggle/working
pwd
ls -lah
# Si ya tienes un notebook en working, conserva copia antes de editar:
cp proyecto.ipynb proyecto.respaldo.ipynb
# Un agente puede modificar proyecto.ipynb mediante herramientas autorizadas.
# Para ejecutar un notebook sin interfaz, si nbconvert está disponible:
python -m jupyter nbconvert --to notebook --execute proyecto.ipynb --output proyecto_verificado.ipynb
```

Los archivos de `/kaggle/input` suelen ser de **solo lectura**: copia lo que vayas a cambiar a `/kaggle/working`. Las modificaciones a un `.ipynb` hechas por SSH **no actualizan automáticamente las celdas abiertas ni crean una nueva versión publicada en la interfaz de Kaggle**. Revisa resultados y guarda/exporta tu versión de forma explícita. No ejecutes sin revisión código sugerido por un agente.

## Límites y seguridad

- Sólo actúa sobre el entorno de Kaggle al que el propietario permite acceso; respeta las políticas, duración y cuotas de Kaggle.
- Ni ChatGPT ni el agente obtienen acceso directo por el hecho de ejecutar este notebook: **necesitan cliente SSH, acceso al VPS y credenciales**. No es MCP ni API de ChatGPT.
- El VPS solicita publicar `0.0.0.0:2223`; protege el puerto con firewall y autenticación fuerte. Verifica huellas SSH; no publiques claves, contraseñas, tokens ni logs de conexión.
- Kaggle puede detener el runtime; una sesión SSH abierta no lo convierte en un servidor permanente. Pulsa Detener/Interruptar en la celda de KagSSH para terminar el proceso.
- El proyecto necesita todavía una prueba real de integración Kaggle Secrets + VPS + SSH/PTY/SFTP. `-check` valida la configuración, **no** certifica la conexión.

Documentación completa: `README.md` y `USO_Y_CONFIGURACION.md` del repositorio [ThowiLabs/kagssh-go](https://github.com/ThowiLabs/kagssh-go).